In [50]:
import pandas as pd
import numpy as np

df = pd.read_csv("used_car_price_prediction_cleaned.csv")

print(df.shape)
df.head()

(1024, 15)


,car_id,brand,model,year,mileage_km,engine_size_cc,fuel_type,transmission,owner_count,condition,city,service_history,accident_history,features_count,price_usd
0,1,Volkswagen,Tiguan,2021.0,56398.0,2200.0,Petrol,Manual,1.0,Excellent,Sialkot,Full,No Accident,17.0,19910
1,2,Suzuki,Alto,2009.0,328090.0,800.0,CNG,CVT,4.0,Fair,Karachi,NaN,Minor Accident,11.0,2000
2,3,Audi,A6,2015.0,253548.0,2000.0,Petrol,DCT,2.0,Poor,Gujranwala,Full,No Accident,20.0,4460
3,4,Toyota,Fortuner,2006.0,350000.0,2500.0,Petrol,Automatic,1.0,Poor,NaN,Partial,Minor Accident,6.0,2000
4,5,Kia,Picanto,2014.0,157434.0,800.0,Petrol,CVT,1.0,Good,NaN,Full,No Accident,10.0,3450


In [51]:
X = df.drop(columns=['price_usd', 'car_id'])
y = df['price_usd']

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (1024, 13)
y Shape: (1024,)


In [52]:
categorical_features = [
    'brand',
    'model',
    'fuel_type',
    'transmission',
    'condition',
    'city',
    'service_history',
    'accident_history'
]

numeric_features = [
    'year',
    'mileage_km',
    'engine_size_cc',
    'owner_count',
    'features_count'
]

In [53]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (819, 13)
Testing: (205, 13)


In [54]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

In [55]:
from sklearn.ensemble import RandomForestRegressor

model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ))
])

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [56]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    'Actual Price': y_test.values,
    'Predicted Price': y_pred
})

results.head(10)

,Actual Price,Predicted Price
0,2000,2044.1
1,11260,12953.4
2,14590,15913.2
3,3260,4880.2
4,2000,4400.4
5,20240,21483.0
6,8680,11729.9
7,6090,7049.5
8,2270,2146.5
9,6210,6567.6


In [57]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R² Score:", r2)

MAE: 1875.823414634146
RMSE: 3354.174271560737
R² Score: 0.8792105813843183


In [58]:
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    ExtraTreesRegressor
)

from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd

models = {
    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        random_state=42
    ),

    "Extra Trees": ExtraTreesRegressor(
        n_estimators=100,
        random_state=42
    )
}

results = []

for name, regressor in models.items():

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    results.append({
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

comparison = pd.DataFrame(results)

comparison

,Model,MAE,RMSE,R²
0,Random Forest,1875.823415,3354.174272,0.879211
1,Gradient Boosting,1893.932042,3390.544844,0.876577
2,Extra Trees,1839.576098,3373.829071,0.877791


In [59]:
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

cv_results = []

for name, regressor in models.items():

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', regressor)
    ])

    scores = cross_val_score(
        pipeline,
        X,
        y,
        cv=5,
        scoring='neg_mean_absolute_error',
        n_jobs=-1
    )

    mae_scores = -scores

    cv_results.append({
        'Model': name,
        'Average CV MAE': mae_scores.mean(),
        'Std': mae_scores.std()
    })

cv_comparison = pd.DataFrame(cv_results)

cv_comparison

,Model,Average CV MAE,Std
0,Random Forest,2054.801429,267.765220
1,Gradient Boosting,2104.436612,249.222224
2,Extra Trees,1913.272125,315.273646


In [60]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.pipeline import Pipeline
import joblib

final_model = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', ExtraTreesRegressor(
        n_estimators=100,
        random_state=42
    ))
])

# සියලු cleaned data භාවිතා කර final model එක train කරනවා
final_model.fit(X, y)

# Model එක save කරනවා
joblib.dump(final_model, "car_price_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [61]:
pip install shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [62]:
import shap

print("SHAP installed successfully!")
print(shap.__version__)

SHAP installed successfully!
0.52.0


In [63]:
import shap
import pandas as pd
import numpy as np

preprocessor_fitted = final_model.named_steps['preprocessor']
extra_trees_model = final_model.named_steps['regressor']

sample_car = X_test.iloc[[0]]

sample_car

,brand,model,year,mileage_km,engine_size_cc,fuel_type,transmission,owner_count,condition,city,service_history,accident_history,features_count
525,Honda,Accord,2012.0,350000.0,1500.0,Petrol,Manual,1.0,Fair,Sialkot,Full,No Accident,8.0


In [64]:
sample_transformed = preprocessor_fitted.transform(sample_car)

if hasattr(sample_transformed, "toarray"):
    sample_transformed = sample_transformed.toarray()

feature_names = preprocessor_fitted.get_feature_names_out()

print("Number of features:", len(feature_names))

Number of features: 144


In [65]:
explainer = shap.TreeExplainer(extra_trees_model)

shap_values = explainer.shap_values(sample_transformed)

prediction = final_model.predict(sample_car)[0]

print("Predicted Price: $", round(prediction, 2))

Predicted Price: $ 2000.0


In [66]:
explanation = pd.DataFrame({
    'Feature': feature_names,
    'SHAP Value': np.array(shap_values).reshape(-1)
})

explanation['Impact'] = explanation['SHAP Value'].abs()

top_factors = explanation.sort_values(
    by='Impact',
    ascending=False
).head(10)

top_factors[['Feature', 'SHAP Value']]

,Feature,SHAP Value
0,num__year,-3773.463297
1,num__mileage_km,-917.234752
125,cat__condition_Excellent,-906.930906
2,num__engine_size_cc,-451.857134
127,cat__condition_Good,-277.163654
18,cat__brand_Suzuki,240.782574
71,cat__model_Land Cruiser,-240.403936
4,num__features_count,-156.415680
3,num__owner_count,123.789000
33,cat__model_Accord,103.638961


In [67]:
def get_price_explanation(car, final_model, top_n=5):
    import shap
    import numpy as np
    import pandas as pd

    preprocessor = final_model.named_steps['preprocessor']
    regressor = final_model.named_steps['regressor']

    transformed = preprocessor.transform(car)

    if hasattr(transformed, "toarray"):
        transformed = transformed.toarray()

    feature_names = preprocessor.get_feature_names_out()

    explainer = shap.TreeExplainer(regressor)
    shap_values = explainer.shap_values(transformed)

    shap_values = np.array(shap_values).reshape(-1)

    # Original feature එකකට SHAP values එකතු කරනවා
    grouped = {}

    for feature, value in zip(feature_names, shap_values):

        clean_name = feature.replace("num__", "").replace("cat__", "")

        original_feature = None

        # Numeric features
        if clean_name in numeric_features:
            original_feature = clean_name

        # Categorical features
        else:
            for col in categorical_features:
                if clean_name.startswith(col + "_"):
                    original_feature = col
                    break

        if original_feature:
            grouped[original_feature] = (
                grouped.get(original_feature, 0) + value
            )

    # වැඩිපුරම බලපෑ features
    sorted_features = sorted(
        grouped.items(),
        key=lambda x: abs(x[1]),
        reverse=True
    )[:top_n]

    explanations = []

    nice_names = {
        "year": "Model Year",
        "mileage_km": "Mileage",
        "engine_size_cc": "Engine Size",
        "owner_count": "Previous Owners",
        "features_count": "Features",
        "brand": "Brand",
        "model": "Model",
        "fuel_type": "Fuel Type",
        "transmission": "Transmission",
        "condition": "Vehicle Condition",
        "city": "City",
        "service_history": "Service History",
        "accident_history": "Accident History"
    }

    for feature, impact in sorted_features:

        value = car.iloc[0][feature]
        name = nice_names.get(feature, feature)

        if impact > 0:
            direction = "↑ Increased the estimated price"
        else:
            direction = "↓ Reduced the estimated price"

        explanations.append({
            "feature": name,
            "value": value,
            "impact": round(float(impact), 2),
            "direction": direction
        })

    return explanations

In [68]:
explanations = get_price_explanation(
    sample_car,
    final_model
)

for item in explanations:
    print(
        item["feature"],
        ":",
        item["value"],
        "→",
        item["direction"],
        "($",
        abs(item["impact"]),
        ")"
    )

Model Year : 2012.0 → ↓ Reduced the estimated price ($ 3773.46 )
Vehicle Condition : Fair → ↓ Reduced the estimated price ($ 1268.93 )
Mileage : 350000.0 → ↓ Reduced the estimated price ($ 917.23 )
Model : Accord → ↓ Reduced the estimated price ($ 501.02 )
Engine Size : 1500.0 → ↓ Reduced the estimated price ($ 451.86 )


In [69]:
import joblib

joblib.dump(final_model, "car_price_model.pkl")

print("Model saved successfully!")

Model saved successfully!


In [70]:
import os

print(os.getcwd())
print(os.path.exists("car_price_model.pkl"))

C:\Users\USER\Desktop\CARNEW
True


In [72]:
import requests

car = {
    "brand": "Honda",
    "model": "Accord",
    "year": 2020,
    "mileage_km": 60000,
    "engine_size_cc": 1500,
    "fuel_type": "Petrol",
    "transmission": "Automatic",
    "owner_count": 1,
    "condition": "Excellent",
    "city": "Colombo",
    "service_history": "Yes",
    "accident_history": "No",
    "features_count": 8
}

response = requests.post(
    "http://127.0.0.1:5000/predict",
    json=car
)
print("Status Code:", response.status_code)
print("Response:")
print(response.text)

Status Code: 503
Response:
<!DOCTYPE html PUBLIC "-//W3C//DTD HTML 4.01//EN" "http://www.w3.org/TR/html4/strict.dtd">
<html><head>
<meta type="copyright" content="Copyright (C) 1996-2023 The Squid Software Foundation and contributors">
<meta http-equiv="Content-Type" CONTENT="text/html; charset=utf-8">
<title>ERROR: The requested URL could not be retrieved</title>
<style type="text/css"><!-- 
 /*
 * Copyright (C) 1996-2023 The Squid Software Foundation and contributors
 *
 * Squid software is distributed under GPLv2+ license and includes
 * contributions from numerous individuals and organizations.
 * Please see the COPYING and CONTRIBUTORS files for details.
 */

/*
 Stylesheet for Squid Error pages
 Adapted from design by Free CSS Templates
 http://www.freecsstemplates.org
 Released for free under a Creative Commons Attribution 2.5 License
*/

/* Page basics */
* {
	font-family: verdana, sans-serif;
}

html body {
	margin: 0;
	padding: 0;
	background: #efefef;
	font-size: 12px;
	colo

In [73]:
import requests

car = {
    "brand": "Honda",
    "model": "Accord",
    "year": 2020,
    "mileage_km": 60000,
    "engine_size_cc": 1500,
    "fuel_type": "Petrol",
    "transmission": "Automatic",
    "owner_count": 1,
    "condition": "Excellent",
    "city": "Colombo",
    "service_history": "Yes",
    "accident_history": "No",
    "features_count": 8
}

session = requests.Session()
session.trust_env = False

response = session.post(
    "http://127.0.0.1:5000/predict",
    json=car
)

print("Status Code:", response.status_code)
print("Response:", response.text)

Status Code: 200
Response: {
  "predicted_price": 12365.3
}



In [74]:
import requests

car = {
    "brand": "Honda",
    "model": "Accord",
    "year": 2020,
    "mileage_km": 60000,
    "engine_size_cc": 1500,
    "fuel_type": "Petrol",
    "transmission": "Automatic",
    "owner_count": 1,
    "condition": "Excellent",
    "city": "Colombo",
    "service_history": "Yes",
    "accident_history": "No",
    "features_count": 8
}

session = requests.Session()
session.trust_env = False

response = session.post(
    "http://127.0.0.1:5000/predict",
    json=car
)

print("Status Code:", response.status_code)
print(response.json())

Status Code: 200
{'explanations': [{'direction': 'increased', 'feature': 'Vehicle Condition', 'message': 'Increased the estimated value', 'value': 'Excellent'}, {'direction': 'increased', 'feature': 'Model Year', 'message': 'Increased the estimated value', 'value': 2020.0}, {'direction': 'decreased', 'feature': 'Model', 'message': 'Reduced the estimated value', 'value': 'Accord'}, {'direction': 'increased', 'feature': 'Mileage', 'message': 'Increased the estimated value', 'value': 60000.0}, {'direction': 'decreased', 'feature': 'Engine Size', 'message': 'Reduced the estimated value', 'value': 1500.0}], 'predicted_price': 12365.3}
